# FLAN-T5-small + LoRA — test-input generator (training notebook, bundle v3)

**Status: prepared, NOT run.** Each step needs separate owner approval first:

* **G-A** upload bundle v3 to `MyDrive/FYP_AI_Training/test_input_v3_bundle` (done for v3 only after approval)
* **G-B** in Colab: install the pinned `transformers`, `accelerate`, `peft` and download
  `google/flan-t5-small` at the pinned revision, then train (all pins in `test_input_lora_config.json`)
* **G-C** download `test_input_v1_output/final/test_input_generator_v1` to the local `models/` folder

Reliability: cell 1 mounts Drive (interactive sign-in, nothing stored here) and verifies every
bundle file's SHA-256; cell 3b refuses to run without a suitable GPU; cell 6 keeps verified
checkpoints on Drive every 500 steps and **resumes** (optimizer, scheduler, RNG, global step) after a
disconnect — it never silently restarts from step 0. Just re-run all cells after a disconnect.

The held-out validation problems and the test split are never uploaded.

In [ ]:
# 1. Mount Google Drive, verify the bundle, approval guards ----------------------
import json, hashlib, pathlib, sys
sys.dont_write_bytecode = True               # never write __pycache__ into the Drive bundle
from google.colab import drive
drive.mount('/content/drive')                # interactive Google sign-in; no credentials are stored in this notebook
BUNDLE = pathlib.Path('/content/drive/MyDrive/FYP_AI_Training/test_input_v3_bundle')
OUT    = pathlib.Path('/content/drive/MyDrive/FYP_AI_Training/test_input_v1_output')
sys.path.insert(0, str(BUNDLE))
from colab_training_utils import verify_bundle, sha256_file
BUNDLE_MANIFEST = verify_bundle(BUNDLE)      # every listed file present with matching size + SHA-256, else stops
print('bundle verified:', len(BUNDLE_MANIFEST['files']), 'files')
CONFIG = json.loads((BUNDLE / 'test_input_lora_config.json').read_text())
assert CONFIG['base_model']['revision'], 'REFUSED: pin base_model.revision (approved commit hash) first'
assert CONFIG['peft_version'], 'REFUSED: pin peft_version (approved exact version) first'
PINS = CONFIG['pinned_packages']
assert all(PINS.get(p) for p in ('peft', 'transformers', 'accelerate')), 'REFUSED: pin every package first'
assert PINS['peft'] == CONFIG['peft_version'], 'REFUSED: inconsistent peft pin'
assert sys.version_info >= tuple(int(x) for x in CONFIG['runtime']['python_min'].split('.')), f'REFUSED: Python {sys.version_info[:2]} < {CONFIG["runtime"]["python_min"]}'
FINAL = OUT / CONFIG['export']['final_dir']
assert not FINAL.exists(), f'REFUSED: final export folder {FINAL} exists (never overwritten)'

In [ ]:
# 2. Install the pinned packages (G-B) ----------------------------------------
# Restart the runtime if transformers was already imported before this cell.
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       *(f'{name}=={PINS[name]}' for name in ('transformers', 'accelerate', 'peft'))])

In [ ]:
# 3. Load the verified data and repeat the problem-level split guards -----------
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
sha256 = sha256_file
for name in ('train.jsonl', 'validation_dev.jsonl'):
    assert sha256(BUNDLE / name) == manifest['files'][name]['sha256'], f'hash mismatch: {name}'
train_rows = [json.loads(l) for l in open(BUNDLE / 'train.jsonl', encoding='utf-8')]
eval_rows  = [json.loads(l) for l in open(BUNDLE / 'validation_dev.jsonl', encoding='utf-8')]
assert {r['split'] for r in train_rows} == {'train'}
assert {r['split'] for r in eval_rows} == {'validation'} and {r['subset'] for r in eval_rows} == {'dev'}
assert not ({r['problem_id'] for r in train_rows} & {r['problem_id'] for r in eval_rows})
print(len(train_rows), 'train pairs;', len(eval_rows), 'validation-dev pairs')

In [ ]:
# 3b. GPU preflight (before any model loading) -----------------------------------
import torch
from colab_training_utils import gpu_preflight
C = CONFIG['checkpointing']
GPU = gpu_preflight(torch, C['min_total_vram_gib'], C['min_free_vram_gib'])   # stops clearly without a suitable GPU
print(json.dumps(GPU, indent=1))
print('configuration (unchanged): batch', CONFIG['training']['per_device_train_batch_size'],
      '| grad accumulation', CONFIG['training']['gradient_accumulation_steps'], '|', CONFIG['training']['precision'])

In [ ]:
# 4. Model, tokenizer and LoRA -------------------------------------------------
import torch, random, numpy as np, transformers, peft, accelerate
for module in (transformers, accelerate, peft):
    assert module.__version__ == PINS[module.__name__], f'REFUSED: {module.__name__} {module.__version__} != pinned {PINS[module.__name__]} (restart runtime)'
from packaging.version import Version
assert Version(torch.__version__.split('+')[0]) >= Version(CONFIG['runtime']['torch_min']), f'REFUSED: torch {torch.__version__} too old'
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType
SEED = CONFIG['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
REPO, REV = CONFIG['base_model']['repo'], CONFIG['base_model']['revision']
tokenizer = AutoTokenizer.from_pretrained(REPO, revision=REV)
base = AutoModelForSeq2SeqLM.from_pretrained(REPO, revision=REV)
L = CONFIG['lora']
model = get_peft_model(base, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=L['r'], lora_alpha=L['lora_alpha'],
                                        lora_dropout=L['lora_dropout'], bias=L['bias'],
                                        target_modules=L['target_modules']))
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable == L['expected_trainable_parameters'], trainable

In [ ]:
# 5a. Target round-trip: every label must decode back to its intended input text --
# Chain checked per row: target -> codec decode (intended input text, canonical)
# -> codec re-encode == target, and target -> PINNED base-tokenizer ids -> decode
# == target -> codec decode == the same intended text. Any failure stops training.
sys.path.insert(0, str(BUNDLE))
from test_input_codec import CodecError, canonical_form, decode_input, encode_input
def target_round_trip_failures(rows):
    failures = []
    for r in rows:
        try:
            text = decode_input(r['target'])                       # intended input text
            ok = canonical_form(text) == text and encode_input(text) == r['target']
            ids = tokenizer(text_target=r['target'], truncation=False)['input_ids']
            seen = tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            ok = ok and seen == r['target'] and decode_input(seen) == text
        except CodecError:
            ok = False
        if not ok:
            failures.append((r['problem_id'], r['case_index']))
    return failures
for name, rows in (('train', train_rows), ('validation_dev', eval_rows)):
    bad = target_round_trip_failures(rows)
    assert not bad, f'REFUSED: {len(bad)} {name} targets do not round-trip, e.g. {bad[:5]}'
print('target round-trip OK under the pinned base tokenizer:', len(train_rows) + len(eval_rows), 'rows')

In [ ]:
# 5. Tokenise without truncation (rows over the limits were removed locally) ---
D = CONFIG['data']
def encode(rows):
    out = []
    for r in rows:
        src = tokenizer(r['prompt'], truncation=False)['input_ids']
        tgt = tokenizer(text_target=r['target'], truncation=False)['input_ids']
        assert len(src) <= D['max_source_tokens'] and len(tgt) <= D['max_target_tokens'] + 1, r['problem_id']
        out.append({'input_ids': src, 'attention_mask': [1] * len(src), 'labels': tgt})
    return out
train_ds, eval_ds = encode(train_rows), encode(eval_rows)
src_len = sorted(len(x['input_ids']) for x in train_ds); tgt_len = sorted(len(x['labels']) for x in train_ds)
print('source tokens median/max', src_len[len(src_len)//2], src_len[-1], '| target median/max', tgt_len[len(tgt_len)//2], tgt_len[-1])

In [ ]:
# 6. Train with verified Drive checkpoints and genuine resume ----------------------
import math, time
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
from colab_training_utils import (run_fingerprint, check_drive_space, restore_for_resume,
                                  make_drive_sync_callback)
T = CONFIG['training']
FINGERPRINT = run_fingerprint(CONFIG, BUNDLE_MANIFEST)     # model revision + pins + data hashes + configs
DRIVE_CKPT, LOCAL_CKPT = OUT / C['drive_dir'], pathlib.Path(C['local_dir'])
check_drive_space(pathlib.Path('/content/drive/MyDrive'), int(C['required_drive_free_gib'] * 2 ** 30))
LOCAL_CKPT.mkdir(parents=True, exist_ok=True)
# None only when no checkpoint was ever written; raises (never restarts at step 0) when checkpoints
# exist but none is complete + compatible. Restores the latest and the best checkpoint locally.
RESUME = restore_for_resume(DRIVE_CKPT, LOCAL_CKPT, FINGERPRINT)
assert RESUME is not None or not any(LOCAL_CKPT.glob('checkpoint-*')), \
    'REFUSED: local checkpoints exist but Drive has none; restart the runtime for a clean fresh start'
print('fresh start' if RESUME is None else f'resuming from {RESUME.name}')
args = Seq2SeqTrainingArguments(
    output_dir=str(LOCAL_CKPT), seed=SEED, learning_rate=T['learning_rate'],
    lr_scheduler_type=T['lr_scheduler_type'], weight_decay=T['weight_decay'],
    # transformers 5.22.2 has no warmup_ratio: a float in [0, 1) passed as warmup_steps is a ratio
    # (get_warmup_steps -> ceil(total_steps * 0.03)).
    warmup_steps=T['warmup_ratio'],
    per_device_train_batch_size=T['per_device_train_batch_size'],
    per_device_eval_batch_size=T['per_device_eval_batch_size'],
    gradient_accumulation_steps=T['gradient_accumulation_steps'], num_train_epochs=T['num_train_epochs'],
    eval_strategy=T['eval_strategy'], save_strategy=T['save_strategy'],
    eval_steps=T['eval_steps'], save_steps=T['save_steps'],
    load_best_model_at_end=T['load_best_model_at_end'], metric_for_best_model=T['metric_for_best_model'],
    save_total_limit=T['save_total_limit'], fp16=False, bf16=False, report_to=[])
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                         data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
                         callbacks=[make_drive_sync_callback(TrainerCallback, DRIVE_CKPT, FINGERPRINT,
                                                             C['keep_last_on_drive'])])
total_steps = math.ceil(len(train_ds) / (T['per_device_train_batch_size'] * T['gradient_accumulation_steps'])) * T['num_train_epochs']
print('planned optimizer steps', total_steps, '| warmup steps', args.get_warmup_steps(total_steps),
      '| checkpoint every', T['save_steps'], 'steps')
started = time.time()
train_result = trainer.train(resume_from_checkpoint=str(RESUME) if RESUME else None)
wall = time.time() - started
eval_metrics = trainer.evaluate()
print(train_result.metrics, eval_metrics, f'{wall:.0f}s this session')

In [ ]:
# 7. Export (separate folder): adapter + merged plain-T5 checkpoint + manifest -----
E = CONFIG['export']
FINAL.mkdir(parents=True)                                  # refused in cell 1 if it already exists
model.save_pretrained(FINAL / E['adapter_dir'])
merged = model.merge_and_unload()
merged.save_pretrained(FINAL / E['merged_dir'])            # transformers 5.x writes model.safetensors
tokenizer.save_pretrained(FINAL / E['merged_dir'])
weights = FINAL / E['merged_dir'] / 'model.safetensors'
assert weights.is_file(), 'REFUSED: merged model.safetensors missing'
training_manifest = {
    'name': CONFIG['name'], 'base_model': REPO, 'base_model_revision': REV,
    'model_safetensors_sha256': sha256(weights),
    'dataset_manifest_sha256': hashlib.sha256((BUNDLE / 'manifest.json').read_bytes()).hexdigest(),
    'data_files': {n: manifest['files'][n] for n in ('train.jsonl', 'validation_dev.jsonl')},
    'config': CONFIG, 'trainable_parameters': trainable, 'run_fingerprint': FINGERPRINT,
    'resumed_from': RESUME.name if RESUME else None, 'best_checkpoint': trainer.state.best_model_checkpoint,
    'global_step': trainer.state.global_step,
    'train_metrics': train_result.metrics, 'eval_metrics': eval_metrics, 'wall_seconds_last_session': round(wall, 1),
    'gpu': GPU,
    'versions': {'python': sys.version.split()[0], 'torch': torch.__version__, 'transformers': transformers.__version__,
             'accelerate': accelerate.__version__, 'peft': peft.__version__},
}
(FINAL / E['merged_dir'] / 'TRAINING_MANIFEST.json').write_text(json.dumps(training_manifest, indent=1))
print(json.dumps({k: training_manifest[k] for k in ('model_safetensors_sha256', 'global_step', 'eval_metrics')}, indent=1))

In [ ]:
# 8. Sanity check: decode a few validation-dev generations with the shared codec --
sys.path.insert(0, str(BUNDLE))
from test_input_codec import decode_model_output
merged.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
for r in eval_rows[:5]:
    enc = tokenizer(r['prompt'], return_tensors='pt').to(merged.device)
    out = merged.generate(**enc, max_new_tokens=256, do_sample=True, top_p=0.95, temperature=0.9,
                          num_return_sequences=3)
    texts = [decode_model_output(tokenizer.decode(o, skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=False))[0] for o in out]
    print(r['problem_id'], texts)

Validity, diversity and boundary metrics are computed **locally** after G-C with
`dev_tools/evaluate_test_inputs.py run --method model` (validation dev, then held-out),
so the same validators judge the model and the deterministic baseline.